# Day 4: Talking to an LLM

For three days we've built graphs with plain Python. Today we finally bring in a real language model, OpenAI's `gpt-4.1-nano`.

I want to set the right expectation first. An LLM call is not some special graph feature. It's a function call that takes text and gives back text. So once it's working on its own, putting it inside a node is the easy part. Most of today is about understanding how to *talk* to the model properly.

By the end of today you'll be able to:

- keep your API key safe in a `.env` file
- call the model and read its answer
- understand the three kinds of chat messages: system, human and AI
- see for yourself why a chatbot needs to resend the conversation every time
- stream an answer word by word
- put an LLM inside a graph node

About cost: `gpt-4.1-nano` is the smallest OpenAI model. Running this whole notebook costs less than a rupee.

## 1. Where the API key lives

Let me start with a rule I'm strict about in class: **never paste an API key into a notebook.** Notebooks get shared, pushed to GitHub, emailed around. Bots scan GitHub for keys all day, and a leaked key can be used to run up a bill on your account within minutes.

Instead we keep the key in a small text file called `.env` in the main folder of this repo:

```
OPENAI_API_KEY=sk-...your key...
OPENAI_MODEL=gpt-4.1-nano
```

If you don't have it yet, copy `.env.example` to `.env` and paste your key in. The `.env` file is listed in `.gitignore`, so Git never uploads it.

The `python-dotenv` package reads that file and loads the values as environment variables. Think of it as the notebook quietly looking in your drawer for the key, instead of you writing the key on the whiteboard.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

In [ ]:
# find_dotenv looks in this folder first, then the parent folders,
# so it finds the .env in the repo root even though we're inside Day04/
load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("I couldn't find OPENAI_API_KEY. Copy .env.example to .env in the repo root and add your key.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
print("Key loaded. Using model:", MODEL)

Notice we only printed the model name, not the key. Get into the habit of never printing secrets, even to check they loaded.

## 2. Your first LLM call

`ChatOpenAI` is LangChain's wrapper around OpenAI's chat models. We'll set `temperature=0`. Temperature is like a creativity knob. At 0 the model gives more stable, predictable answers, which is what you want while learning. Higher values make it more varied and creative.

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model=MODEL, temperature=0)

### Predict before you run

We'll call `llm.invoke("In one sentence, what is LangGraph?")`. What do you think comes back: a plain string, or something else?

<details>
<summary>Click to see the answer</summary>

Something else: an `AIMessage` object. The actual text is inside `.content`.

It's a whole object rather than a string because the model sends back more than text: token counts, the reason it stopped, and later (on Day 6) requests to use tools.
</details>

In [ ]:
response = llm.invoke("In one sentence, what is LangGraph?")

print(type(response).__name__)
print(response.content)

Let's peek at one of those extras, the token usage. Tokens are roughly pieces of words, and they're what you pay for.

In [ ]:
print(response.usage_metadata)

## 3. Messages: who said what

When you chat with ChatGPT, the model isn't receiving one string. It's receiving a list of messages, and each message says **who** is speaking. There are three you'll use all the time:

- **SystemMessage**: instructions from you, the developer. "You are a helpful tutor. Keep answers short." The user never sees this. Think of it as the briefing a new employee gets before meeting customers.
- **HumanMessage**: what the user typed.
- **AIMessage**: what the model replied.

(There's a fourth, `ToolMessage`, which we'll meet on Day 6.)

The system message is surprisingly powerful. Watch how it completely changes the style of the answer.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
messages = [
    SystemMessage(content="You are a cricket commentator. Answer in 2 short, excited lines."),
    HumanMessage(content="Explain what a Python function is."),
]

reply = llm.invoke(messages)
reply.pretty_print()

### Your turn

Change the system message to a different personality, maybe a strict school teacher, or a chef explaining everything with cooking examples. Keep the question the same and see how much the answer changes.

In [ ]:
messages = [
    SystemMessage(content="You are a chef. Explain everything using kitchen examples. Max 3 lines."),  # change this
    HumanMessage(content="Explain what a Python function is."),
]
print(llm.invoke(messages).content)

There's also a shortcut you'll see in a lot of code. Instead of message objects, you can write tuples of `(role, text)`. Same thing, less typing.

In [ ]:
reply = llm.invoke([
    ("system", "Reply with exactly one word, nothing else."),
    ("user", "How do you feel about Monday mornings?"),
])
print(reply.content)

## 4. The model has no memory

This is the single most important thing to understand before we build chatbots tomorrow, so let's prove it rather than just say it.

Every call to the model is completely independent. It's like phoning a call centre where a different person picks up every time. Unless you repeat the earlier conversation, they have no idea what you said before.

### Predict before you run

First we tell the model our name. Then, in a **separate** call, we ask it what our name is. Will it know?

<details>
<summary>Click to see the answer</summary>

No. The second call has no idea the first one ever happened. The model will say something like "I don't know your name."
</details>

In [ ]:
llm.invoke([HumanMessage(content="Hi! My name is Asha.")])

answer = llm.invoke([HumanMessage(content="What is my name?")])
print("Without history:", answer.content)

Now let's send the earlier conversation along with the new question.

In [ ]:
history = [
    HumanMessage(content="Hi! My name is Asha."),
    AIMessage(content="Nice to meet you, Asha!"),
    HumanMessage(content="What is my name?"),
]
print("With history:", llm.invoke(history).content)

So here's the secret of every chatbot you've ever used: **a chatbot is just a list of messages that keeps growing, and the whole list is sent to the model every single time.** The "memory" is something *we* manage, not the model.

Managing that list by hand gets messy quickly. Tomorrow, LangGraph will manage it for us.

## 5. Streaming: words appear as they're written

When ChatGPT "types" its answer, it's streaming. The model sends small pieces (chunks) as soon as they're ready, instead of making you wait for the full answer. For long answers this makes your app feel much faster, even though the total time is the same.

In [ ]:
for chunk in llm.stream("Count from 1 to 10 in words, separated by spaces."):
    print(chunk.content, end="", flush=True)
print()

## 6. Putting the LLM inside a graph

Now let's connect this to what we already know. Remember, a node is just a function. So a node that calls an LLM is just a function that calls an LLM. Same five steps as Day 1.

```
  START --> answer --> END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

In [ ]:
class QAState(TypedDict):
    question: str
    answer: str

In [ ]:
def answer(state: QAState) -> dict:
    # Build the messages, ask the model, and store only the text
    reply = llm.invoke([
        SystemMessage(content="You are a patient teacher. Answer in at most 3 sentences."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": reply.content}

In [ ]:
builder = StateGraph(QAState)

In [ ]:
builder.add_node("answer", answer)

In [ ]:
builder.add_edge(START, "answer")
builder.add_edge("answer", END)

In [ ]:
qa_app = builder.compile()

In [ ]:
result = qa_app.invoke({"question": "Why is the sky blue?"})
print(result["answer"])

That's it. If you understood Day 1, you already understood this.

## 7. Common mistakes

**Mistake 1: "OPENAI_API_KEY not found" even though you created `.env`.**
Usually one of these:
- the file is called `.env.txt` (Windows likes to hide extensions, turn on "show file extensions")
- it's in the wrong folder (it should be in the repo root, next to `README.md`)
- there are spaces or quotes: write `OPENAI_API_KEY=sk-abc`, not `OPENAI_API_KEY = "sk-abc"`
- you created `.env` after starting Jupyter. Restart the kernel and run the setup cell again.

**Mistake 2: Printing the response instead of its text.**
`print(response)` shows the whole object with lots of metadata. You almost always want `response.content`.

In [ ]:
response = llm.invoke("Say hi in Marathi")
print("Whole object:", response)
print()
print("Just the text:", response.content)

**Mistake 3: Storing the AIMessage object when your state expects a string.**
In the `answer` node above we stored `reply.content`, not `reply`. If your `TypedDict` says `answer: str`, store the string. (Tomorrow we'll store whole message objects on purpose, but that's a different kind of state.)

**Mistake 4: A typo in the model name.**
`gpt-4.1-nan0` gives a "model not found" error from OpenAI. If you see a 404 error from the API, check the model name in `.env`.

## 8. Mini project: "Explain like I'm 10"

Two LLM nodes in a row. The first explains a topic for an engineer, the second rewrites that for a 10-year-old. This is the simplest version of something you'll see a lot: one LLM step improving the output of another.

```
  START --> explain --> simplify --> END
```

In [ ]:
class ExplainState(TypedDict):
    topic: str
    explanation: str
    simple: str

In [ ]:
def explain(state):
    reply = llm.invoke(f"Explain '{state['topic']}' in 4 sentences for a software engineer.")
    return {"explanation": reply.content}

In [ ]:
def simplify(state):
    # This node works on what the previous node wrote
    reply = llm.invoke(
        "Rewrite this so a 10-year-old can understand it. Use one fun example. "
        f"Maximum 3 sentences.\n\n{state['explanation']}"
    )
    return {"simple": reply.content}

In [ ]:
builder = StateGraph(ExplainState)

In [ ]:
builder.add_sequence([explain, simplify])

In [ ]:
builder.add_edge(START, "explain")
builder.add_edge("simplify", END)

In [ ]:
eli10 = builder.compile()
show_graph(eli10)

In [ ]:
result = eli10.invoke({"topic": "neural networks"})
print("FOR AN ENGINEER:\n", result["explanation"])
print()
print("FOR A 10-YEAR-OLD:\n", result["simple"])

### Your turn

Try your own topic. Something from your job or studies works best, because you'll be able to judge whether the simple version is actually correct.

In [ ]:
my_topic = "blockchain"      # change this
print(eli10.invoke({"topic": my_topic})["simple"])

## 9. Practice

**Exercise 1.** Build a `translate` graph. The state has `text`, `language` and `translation`. One node asks the model to translate `text` into `language`. Try Hindi and Marathi. Hint: use a system message telling it to reply with only the translation.

**Exercise 2.** Build `START -> write_tweet -> add_hashtags -> END`. The first node writes a short tweet about a `topic` without hashtags. The second asks the model to add exactly 3 hashtags.

In [ ]:
# Solution 1
class TranslateState(TypedDict):
    text: str
    language: str
    translation: str

In [ ]:
def translate(state):
    reply = llm.invoke([
        SystemMessage(content="You are a translator. Reply with only the translation, nothing else."),
        HumanMessage(content=f"Translate into {state['language']}: {state['text']}"),
    ])
    return {"translation": reply.content}

In [ ]:
builder = StateGraph(TranslateState)

In [ ]:
builder.add_node("translate", translate)

In [ ]:
builder.add_edge(START, "translate")
builder.add_edge("translate", END)

In [ ]:
translator = builder.compile()

In [ ]:
for language in ("Hindi", "Marathi"):
    result = translator.invoke({"text": "Learning never stops.", "language": language})
    print(language, ":", result["translation"])

In [ ]:
# Solution 2
class TweetState(TypedDict):
    topic: str
    tweet: str

In [ ]:
def write_tweet(state):
    reply = llm.invoke(f"Write one short tweet (under 200 characters) about {state['topic']}. No hashtags.")
    return {"tweet": reply.content}

In [ ]:
def add_hashtags(state):
    reply = llm.invoke(f"Add exactly 3 relevant hashtags at the end of this tweet. Return only the full tweet.\n\n{state['tweet']}")
    return {"tweet": reply.content}

In [ ]:
builder = StateGraph(TweetState)

In [ ]:
builder.add_sequence([write_tweet, add_hashtags])

In [ ]:
builder.add_edge(START, "write_tweet")
builder.add_edge("add_hashtags", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"topic": "learning LangGraph"})["tweet"])

## Wrapping up

Your key lives in `.env` and gets loaded with `load_dotenv`. `ChatOpenAI` gives you a model, and `.invoke()` returns an `AIMessage` with the text in `.content`. Chat models take a list of messages (system, human, AI), and the most important lesson today is that **the model remembers nothing**. Any memory is the history we choose to send.

An LLM inside a graph is just a node that happens to call the model.

Tomorrow we build a proper chatbot, and LangGraph takes over the boring job of managing that growing list of messages.